In [ ]:
import os
import yaml
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import gc
import json
import itertools

# ============================================================
# [MLP - Multi-Layer Perceptron]
# - sklearn RF와 달리 외삽(extrapolation) 가능
# - GPU(L4) 활용
# - 피처: cpu, memory, duration
# - 타겟: energy_kwh
# ============================================================

# GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA L4


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ============================================================
# Config 로드
# ============================================================
CONFIG_PATH = "/content/config.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

DRIVE_ROOT     = f"/content/drive/MyDrive/{config['project_name']}"
PROCESSED_PATH = os.path.join(DRIVE_ROOT, config['paths']['processed_data'],
                               config['paths']['processed_file'])
MODEL_PATH     = os.path.join(DRIVE_ROOT, config['paths']['models'])
REPORT_PATH    = os.path.join(DRIVE_ROOT, config['paths']['outputs'], "reports")

In [9]:
# ============================================================
# 데이터 로드 + 샘플링 + 증강
# ============================================================
df_full = pd.read_parquet(PROCESSED_PATH)

SAMPLE_SIZE = 2_000_000
df = df_full.sample(n=SAMPLE_SIZE, random_state=config['model']['random_state'])
del df_full; gc.collect()

DURATION_MULTIPLIERS = [1, 2, 4, 6, 12]

augmented_dfs = []
for m in DURATION_MULTIPLIERS:
    df_aug = df.copy()
    df_aug['duration']   = df_aug['duration']   * m
    df_aug['energy_kwh'] = df_aug['energy_kwh'] * m
    augmented_dfs.append(df_aug)

df_aug = pd.concat(augmented_dfs, ignore_index=True)
del augmented_dfs; gc.collect()
print(f"Augmented rows: {len(df_aug):,}")
print(f"Duration range: {df_aug['duration'].min():.0f} ~ {df_aug['duration'].max():.0f} sec")

Augmented rows: 10,000,000
Duration range: 1 ~ 3600 sec


In [10]:
# ============================================================
# 피처 / 타겟 분리
# ============================================================
features = ["cpu", "memory", "duration"]
target   = "energy_kwh"

X = df_aug[features].values.astype(np.float32)
y = df_aug[target].values.astype(np.float32).reshape(-1, 1)

del df_aug; gc.collect()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 8,000,000 | Test: 2,000,000


In [14]:
# ============================================================
# 정규화 (MLP는 스케일링 필수)
# ============================================================
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train)
y_test_sc = scaler_y.transform(y_test)

# Tensor 변환
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train_sc, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test_sc, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test_sc, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True)

# ============================================================
# MLP 모델 정의
# ============================================================
class EnergyMLP(nn.Module):
    """
    에너지 예측 MLP
    - hidden_sizes : 각 은닉층 뉴런 수 리스트 (예: [256, 128, 64])
    - dropout_rate : Dropout 비율
    - use_batchnorm : BatchNorm 사용 여부
    """
    def __init__(self, hidden_sizes, dropout_rate=0.2, use_batchnorm=True):
        super(EnergyMLP, self).__init__()
        layers = []
        in_size = 3

        for h in hidden_sizes:
            layers.append(nn.Linear(in_size, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_size = h

        layers.append(nn.Linear(in_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [ ]:
# ============================================================
# 평가 함수
# ============================================================
def evaluate(model, X_t, y_true, scaler_y):
    model.eval()
    with torch.no_grad():
        y_pred_sc = model(X_t).cpu().numpy()
    y_pred = scaler_y.inverse_transform(y_pred_sc)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, r2, mape

In [ ]:
# ============================================================
# 하이퍼파라미터 조합 정의
# ============================================================
hp_grid = {
    "hidden_sizes" : [[256, 128, 64, 32], [512, 256, 128], [128, 64, 32]],
    "lr"           : [1e-3, 1e-4],
    "batch_size"   : [1024, 4096],
    "dropout_rate" : [0.2, 0.0],
}

keys   = list(hp_grid.keys())
combos = list(itertools.product(*hp_grid.values()))
print(f"\n총 조합 수: {len(combos)}")

EPOCHS    = 20
results   = []
best_mape = float('inf')
best_model_state = None
best_hp = None

In [ ]:
# ============================================================
# 하이퍼파라미터 튜닝 루프
# ============================================================
for i, combo in enumerate(combos):
    hp = dict(zip(keys, combo))
    label = f"hidden={hp['hidden_sizes']} lr={hp['lr']} batch={hp['batch_size']} drop={hp['dropout_rate']}"
    print(f"\n[{i+1}/{len(combos)}] {label}")

    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader  = DataLoader(train_dataset, batch_size=hp['batch_size'], shuffle=True)

    model = EnergyMLP(
        hidden_sizes=hp['hidden_sizes'],
        dropout_rate=hp['dropout_rate']
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=hp['lr'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    best_val_loss = float('inf')
    best_state    = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_test_t), y_test_t).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

    # 최적 가중치로 평가
    model.load_state_dict(best_state)
    rmse, mae, r2, mape = evaluate(model, X_test_t, y_test, scaler_y)

    print(f"  RMSE={rmse:.8f} | R2={r2:.8f} | MAPE={mape:.4f}%")
    results.append({"config": label, "rmse": rmse, "mae": mae, "r2": r2, "mape": mape, **hp})

    if mape < best_mape:
        best_mape        = mape
        best_model_state = best_state
        best_hp          = hp


In [ ]:
# ============================================================
# 최종 비교
# ============================================================
print("\n=== 최종 비교 ===")
df_results = pd.DataFrame(results)
print(df_results[["config", "rmse", "r2", "mape"]].sort_values("mape").to_string(index=False))
print(f"\n최적 config: {best_hp}")
print(f"최적 MAPE : {best_mape:.4f}%")

# ============================================================
# 최적 모델 저장
# ============================================================
best_model = EnergyMLP(
    hidden_sizes=best_hp['hidden_sizes'],
    dropout_rate=best_hp['dropout_rate']
).to(device)
best_model.load_state_dict(best_model_state)

mlp_model_path = os.path.join(MODEL_PATH, config['model']['model_names']['mlp'])
scaler_X_path = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_x'])
scaler_y_path = os.path.join(MODEL_PATH, config['model']['model_names']['scaler_y'])

torch.save(best_model.state_dict(), mlp_model_path)
with open(scaler_X_path, 'wb') as f: pickle.dump(scaler_X, f)
with open(scaler_y_path, 'wb') as f: pickle.dump(scaler_y, f)

print(f"\nSaved MLP : {mlp_model_path}")
print(f"Saved ScalerX : {scaler_X_path}")
print(f"Saved ScalerY : {scaler_y_path}")


[MLP 최종 성능]
  RMSE : 0.00042799
  MAE  : 0.00027157
  R2   : 0.99995863
  MAPE : 26.9279%


In [ ]:
# ============================================================
# phase1_full_results.json 업데이트
# ============================================================
result_file = os.path.join(REPORT_PATH, config['model']['results']['results_json_mlp'])

mlp_results = {
    'best_config' : best_hp,
    'best_mape'   : float(best_mape),
    'all_results' : results,
    'model_file'  : config['model']['model_names']['mlp'],
}

with open(result_file, 'w') as f:
    json.dump(mlp_results, f, indent=2, ensure_ascii=False)

print(f"\nResults saved: {result_file}")
print("Done!")